# 📄 AI Research Paper Assistant

## The system allows users to:
-  Upload a research paper PDF
-  Generate a summary
-  Extract main contributions
-  Explain technical concepts
-  Ask questions about the paper

# Install Libraries

In [ ]:
# Python packages
!pip install -qU \
    PyMuPDF \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    chromadb \
    pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2

In [ ]:
# System dependency required by the Ollama installer
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 164 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (557 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
# Install Ollama (local LLM runtime)
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [ ]:
# Start the Ollama server in the background
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)
print("Ollama server started!")

Ollama server started!


In [ ]:
# Pull the models used in this notebook:
# - qwen2.5:7b        -> chat/generation model
# - nomic-embed-text  -> embedding model for RAG
# !curl http://localhost:11434
!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:7b                 845dbda0ea48    4.7 GB    5 seconds ago             


In [ ]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.4 MB/s eta 0:00:00


# Imports

In [ ]:
# Standard library
import re
import time
import subprocess
import unicodedata

# PDF parsing
import fitz  # PyMuPDF

# LLM client
import ollama

# Text splitting / embeddings / vector store
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_23971/1884054741.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="missing ScriptRunContext! This warning can be ignored when running in bare mode.",
    category=RuntimeWarning
)

# PDF Processing Engineer

### Text Cleaning

In [ ]:
# Extract Text
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text from all pages of a PDF.
    """
    if not pdf_path.endswith('.pdf'):
        raise ValueError("Invalid file format. Only PDF files are supported.")

    document = fitz.open(pdf_path)
    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

In [ ]:
# Clean Text
def clean_pdf_text(page_text: str) -> str:
    """
    Clean extracted PDF text for RAG applications.

    Steps:
    1. Normalize Unicode characters.
    2. Remove page break characters.
    3. Fix hyphenated words split across lines.
    4. Remove tabs.
    5. Remove isolated page numbers.
    6. Strip extra whitespace from each line.
    7. Collapse multiple spaces.
    8. Collapse excessive blank lines.
    """

    text = unicodedata.normalize("NFKC", page_text)
    text = text.replace("\x0c", "")
    text = text.replace("\t", " ")
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    lines = [
        line for line in lines
        if not re.match(r'^(page\s*)?\d+\s*$', line, re.IGNORECASE)
    ]

    text = "\n".join(lines)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
# Remove References
def remove_references(text: str) -> str:
    """
    Remove the References section and everything after it.
    """

    pattern = r"\nreferences\b.*"

    cleaned = re.sub(
        pattern,
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return cleaned

### Chunking

In [ ]:
# Chunking
def split_into_chunks(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(text)

    return chunks

### Run the PDF Pipeline

In [ ]:
# Test PDF Processing
def process_pdf(pdf_path: str):
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_pdf_text(raw_text)
    cleaned_text = remove_references(cleaned_text)
    return raw_text, cleaned_text

raw_text, clean_text = process_pdf("/content/EJ1172284.pdf")
chunks = split_into_chunks(clean_text)
print(f"Extracted {len(chunks)} chunks")

Extracted 46 chunks


# LLM Engineer

### Response Generation

In [ ]:
# Create a reusable function to generate responses using the Qwen model

MODEL_NAME = "qwen2.5:7b"

def generate_response(
    prompt: str,
    temperature: float = 0.3,
    max_tokens: int = 1024
) -> str:
    """
    Generate a response using the Qwen model through Ollama.

    Args:
        prompt: The input prompt for the LLM.
        temperature: Controls randomness of the generated response.
        max_tokens: Maximum number of tokens to generate.

    Returns:
        The generated text response.
    """

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": temperature,
            "num_predict": max_tokens
        }
    )

    return response["message"]["content"]

### Prompts

In [ ]:
ROUTER_PROMPT = """
You are an intent classifier for a Research Paper Assistant. Classify the user's request into exactly ONE category.

Categories:
- SUMMARY: wants an overview/abstract of the paper (e.g. "Summarize this", "What is this paper about?")
- CONTRIBUTIONS: wants the paper's key contributions, novelty, or findings (e.g. "Main contributions", "What did they find?")
- CONCEPTS: wants a technical term or concept explained (e.g. "Explain Transformers", "What is attention?")
- QUESTION: wants a specific fact/detail from the paper (e.g. "What dataset was used?", "What accuracy did they get?")
- GREETING: small talk, greetings, or asking who you are (e.g. "Hi", "What can you do?")

If the request is ambiguous or fits more than one category, prefer QUESTION.
If the request is unrelated to the paper entirely, respond with QUESTION.

Respond with ONLY the category name in uppercase. No punctuation, no explanation, no extra words.

User Request:
{query}
"""

In [ ]:
SUMMARY_PROMPT = """
You are an expert research assistant.

Read the research paper below carefully.

Rules:
- Use ONLY information from the paper.
- Do NOT invent missing information.
- Do NOT use outside knowledge.
- Do NOT include references or citations.
- Do NOT include related work.
- Do NOT include future work.
- If any section is not explicitly mentioned, write:
  Not explicitly mentioned in the paper.

Paper:
{paper}

---
IMPORTANT: Your response MUST follow EXACTLY this structure, with these exact headings,
and nothing else before or after it:

1. Research Objective
- Briefly describe the main problem the paper aims to solve.

2. Methodology
- Explain the proposed method or approach.
- Mention the model, algorithm, or architecture if applicable.

3. Main Findings
- Return 3-5 bullet points.
- Include only the paper's main experimental findings or achievements.

4. Conclusion
- Summarize the authors' final conclusion in one or two sentences.

Now write the summary using EXACTLY the structure above:
"""

In [ ]:
CONTRIBUTIONS_PROMPT = """
You are an expert research analyst. Identify ONLY the original contributions this paper claims to make.

Rules:
- Include only what the authors present as new (their own method, result, dataset, or insight).
- Exclude background information, related work, prior methods, and future work.
- Exclude anything attributed to cited papers rather than this paper's own authors.
- If you are not confident something is an original contribution, do not include it.
- If no clear contributions are stated, respond exactly: No explicit contributions found in the paper.

Paper:
{paper}

---
IMPORTANT: Return your answer as a bulleted list, ordered from most to least significant.
For each contribution, write ONE clear sentence describing what was done and why it matters.
Return ONLY the bulleted list, nothing else before or after it.
"""

In [ ]:
CONCEPTS_PROMPT = """
You are a patient teaching assistant helping a student understand a research paper.

Your task is to answer the user's question by explaining ONLY the concept they asked about.

Rules:
- Focus only on the concept implied by the user's question.
- Base your explanation on how the concept is used in the research paper.
- Do not explain unrelated concepts.
- If the paper does not contain enough information to answer the question, say:
  "The requested concept is not discussed in the provided paper."
- Use simple undergraduate-level language.
- Use at most 3 concise bullet points.
- Avoid mathematical notation unless it is necessary.

User Question:
{question}

Paper:
{paper}

---

Your response MUST use EXACTLY this format:

**Concept:** [Concept Name]

- Bullet 1
- Bullet 2
- Bullet 3
"""

In [ ]:
QA_PROMPT = """
You are answering a question about a research paper using only the retrieved context below.

Rules:
- Answer using ONLY the context provided. Do not use outside knowledge, even if you know the answer.
- If the context only partially answers the question, answer what you can and note what's missing.
- If the answer is not present in the context at all, reply EXACTLY: I couldn't find this information in the uploaded paper.
- Keep the answer concise and directly address the question — don't restate the full context.
- Do not fabricate numbers, names, or details not present in the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
GREETING_PROMPT = """
You are an AI assistant designed to greet users.
Respond with a friendly greeting and offer help.

Examples:
- Hello! How can I assist you today?
- Hi there! What can I do for you?
- Greetings! I'm here to help with your research paper.
"""

### Query Router

In [ ]:
def classify_request(user_query: str):

    prompt = ROUTER_PROMPT.format(
        query=user_query
    )

    intent = generate_response(
        prompt,
        temperature=0.1,
        max_tokens=10
    )

    return intent.strip().upper()

# RAG Engineer

### Embeddings & Vector Store

In [ ]:
# 4.1 Load Embedding Model
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Ollama Embedding model loaded successfully!")


Ollama Embedding model loaded successfully!


In [ ]:
# 4.2 Create Embeddings
sample_text = chunks[0] if chunks else "Research paper test text."
sample_embedding = embedding_model.embed_query(sample_text)

print(f"Embeddings generated successfully via Ollama!")
print(f"Vector dimensions: {len(sample_embedding)}")


Embeddings generated successfully via Ollama!
Vector dimensions: 768


In [ ]:
# 4.3 Create FAISS Index
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print(f"FAISS Index created successfully with {vector_store.index.ntotal} chunks!")


FAISS Index created successfully with 46 chunks!


### Retrieval

In [ ]:
def retrieve_context(
    query: str,
    k: int = 3
) -> str:
    """
    Retrieve the most relevant chunks from FAISS.
    """

    docs = vector_store.similarity_search(query, k=k)

    if not docs:
        return ""

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return context

### Task's Function

In [ ]:
def generate_summary(
    user_request: str,
    paper_text: str
) -> str:

    prompt = SUMMARY_PROMPT.format(
        paper=paper_text
    )

    return generate_response(
        prompt,
        temperature=0.2,
        max_tokens=700
    )

In [ ]:
def extract_contributions(
    user_request: str,
    paper_text: str
) -> str:

    prompt = CONTRIBUTIONS_PROMPT.format(
        paper=paper_text
    )

    return generate_response(
        prompt,
        temperature=0.2,
        max_tokens=500
    )

In [ ]:
def explain_concepts(
    user_request: str,
    paper_text: str
) -> str:

    prompt = CONCEPTS_PROMPT.format(
        question=user_request,
        paper=paper_text
    )

    return generate_response(
        prompt,
        temperature=0.2,
        max_tokens=700
    )

In [ ]:
def answer_question(
    question: str,
    k: int = 3
) -> str:

    context = retrieve_context(
        question,
        k=k
    )

    prompt = QA_PROMPT.format(
        context=context,
        question=question
    )

    return generate_response(
        prompt,
        temperature=0,
        max_tokens=300
    )

In [ ]:
import random
def greeting(user_request: str) -> str:
    dynamic_prompt = f"{GREETING_PROMPT}\n\nGenerate a unique greeting. Randomness factor: {random.random()}"
    return generate_response(dynamic_prompt, temperature=0.9, max_tokens=100)

### Routering

In [ ]:
def research_assistant(
    user_request: str,
    paper_text: str,
):
    """
    Main routing function.
    """

    intent = classify_request(user_request)

    print(f"\nDetected Intent: {intent}\n")

    if intent == "GREETING":
        return greeting(user_request)

    if intent == "SUMMARY":

        return generate_summary(
            user_request,
            paper_text
        )

    elif intent == "CONTRIBUTIONS":

        return extract_contributions(
            user_request,
            paper_text
        )

    elif intent == "CONCEPTS":

        return explain_concepts(
            user_request,
            paper_text
        )

    elif intent == "QUESTION":

        return answer_question(
            user_request
        )

    else:

        return "Unable to classify the request."

# T

In [ ]:
query = 'can you summarize the pdf'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: SUMMARY

1. Research Objective
- The study aims to explore how advanced English language learners use mobile devices (smartphones and tablet computers) for autonomous learning of the English language, focusing on their motivations, methods, and perceived benefits.

2. Methodology
- A qualitative research design was employed.
- Semi-structured interviews were conducted with 20 advanced English language learners from a single institution to gather data about their use of mobile devices in language learning.

3. Main Findings
- All interviewees used mobile devices for autonomous English study, primarily for vocabulary acquisition and pronunciation practice.
- The majority reported that using mobile devices made them more willing to learn English and spend more time on it.
- Some learners preferred traditional resources or felt proficient enough not to use their smartphones for certain language skills like grammar.
- Only a few subjects used mobile devices during formal c

In [ ]:
query = 'what is the main contributions?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: CONTRIBUTIONS

- The study highlighted that advanced English learners use mobile devices (smartphones/tablets) for autonomous language learning, indicating a growing trend in technology-assisted education.
- It found that these devices are particularly useful for vocabulary acquisition and pronunciation practice, as they offer quick access to information and context.
- The research noted that some students used their mobile devices more effectively outside of formal classroom settings, suggesting the potential for increased learner autonomy.
- The study revealed a gap in language teachers' utilization of mobile technology, with many not fully leveraging its benefits or understanding its role in modern language learning.
- By interviewing advanced learners, the study provided insights into how and why they use their devices, which can inform better integration strategies by educators.


In [ ]:
query = 'what is index cards apps ?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

Index card apps are mentioned as tools used for developing English vocabulary by student S5.


In [ ]:
query = 'is there aconnection between educational technologies and learner autonomy?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

Yes, according to Benson (2011), there has always been a connection between educational technologies and learner autonomy. Benson suggests that these technologies have often been intended for independent practice. Reinders and White (2016) also emphasize the importance of aligning technology use with the tools, settings, and activities significant to language learners, highlighting the need for individuals to be adept at critical adaptive learning in various settings.


In [ ]:
query = 'hello'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: GREETING

Hello there! Today's weather seems perfect for some creative endeavors—what exciting task would you like assistance with?


In [ ]:
query = 'how to use of mobile technology and mobile devices in language learning'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

The use of mobile technology and mobile devices in language learning allows language learners access to numerous language resources anytime and anywhere. This can include using spare time for practice, searching for vocabulary on social media, and engaging in activities that meet individual needs and goals. Mobile devices enable learners to take control of their learning process. However, the exact methods or specific applications mentioned are not detailed in the provided context.


In [ ]:
query = 'is the use of mobile devices was advised or suggested by the interviewees?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

The use of mobile devices was not explicitly advised or suggested by the interviewees’ teachers during their practical English language classes or any other classes at the university.


# Pipeline & Frontend Engineer